In [0]:
%sql
CREATE WIDGET TEXT target_partition DEFAULT "YYYYMM";

In [0]:
target_yellow_partition = dbutils.widgets.get("target_partition")

In [0]:
print(target_yellow_partition)

In [0]:
from pyspark.sql import functions as F 

base_path = "/Volumes/nyc/default/"
yellow_path = base_path + "yellowtaxi/"
delta_target = "nyc.process_silver.delta_yellow_taxi"

BASE_RULES = {
    "missing_pickup_datetime": F.col("pickup_datetime").isNull(),
    "missing_dropoff_datetime": F.col("dropoff_datetime").isNull(),
    "dropoff_before_pickup": (F.col("pickup_datetime").isNotNull())
    & (F.col("dropoff_datetime").isNotNull())
    & (F.col("dropoff_datetime") < F.col("pickup_datetime")),
    "passenger_count_negative": F.col("passenger_count") < 0,
    "trip_distance_too_small": F.col("trip_distance") < 0.1,
    "total_amount_negative": F.col("total_amount") < 0,
    "invalid_YYYYMM": F.col("YYYYMM").isNotNull()
    & ((F.col("YYYYMM") < 190001) | (F.col("YYYYMM") > 300012)),
    "invalid_fare_amount": F.col("fare_amount").isNotNull() & (F.col("fare_amount") < 2.5),
    
}

SILVER_RULES = {
    "invalid_duration_min": (F.col("pickup_datetime").isNotNull())
    & (F.col("dropoff_datetime").isNotNull())
    & ((F.col("duration_min") < 2.0) | (F.col("duration_min") > 300)), 
    "impossible_speed": (F.col("trip_distance") / (F.col("duration_min") / 60) > 100 )
}


In [0]:
%sql
select * from nyc.process_silver.delta_yellow_taxi limit 2; 

In [0]:
%sql
-- 确保进入正确的 Catalog 和 Schema
USE CATALOG nyc;
CREATE SCHEMA IF NOT EXISTS process_silver;

-- 创建目标 Delta 表
CREATE TABLE IF NOT EXISTS process_silver.delta_yellow_taxi (
    vendor_id STRING,
    pickup_datetime TIMESTAMP,
    dropoff_datetime TIMESTAMP,
    passenger_count BIGINT,
    trip_distance DOUBLE,
    rate_code BIGINT,
    store_and_fwd_flag STRING,
    pickup_longitude DOUBLE,
    pickup_latitude DOUBLE,
    dropoff_longitude DOUBLE,
    dropoff_latitude DOUBLE,
    PULocationID BIGINT,
    DOLocationID BIGINT,
    payment_type STRING,
    fare_amount DOUBLE,
    surcharge DOUBLE,
    mta_tax DOUBLE,
    tip_amount DOUBLE,
    tolls_amount DOUBLE,
    improvement_surcharge DOUBLE,
    congestion_surcharge DOUBLE,
    airport_fee DOUBLE,
    cbd_congestion_fee DOUBLE,
    total_amount DOUBLE,
    YYYY INT,
    YYYYMM INT,
    dq_reason ARRAY<STRING>,
    dq_is_bad BOOLEAN
)
USING DELTA
PARTITIONED BY (${target_partition})
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true',
    'comment' = 'NYC Yellow Taxi Silver Table with DQ markers'
);




In [0]:
from __future__ import annotations

from collections import defaultdict
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

BASE_RULES = {
    "missing_pickup_datetime": F.col("pickup_datetime").isNull(),
    "missing_dropoff_datetime": F.col("dropoff_datetime").isNull(),
    "dropoff_before_pickup": (F.col("pickup_datetime").isNotNull())
    & (F.col("dropoff_datetime").isNotNull())
    & (F.col("dropoff_datetime") < F.col("pickup_datetime")),
    "passenger_count_negative": F.col("passenger_count") < 0,
    "trip_distance_too_small": F.col("trip_distance") < 0.1,
    "total_amount_negative": F.col("total_amount") < 0,
    "invalid_YYYYMM": F.col("YYYYMM").isNotNull()
    & ((F.col("YYYYMM") < 190001) | (F.col("YYYYMM") > 300012)),
    "missing_fare_amount": F.col("fare_amount").isNull(),
    "invalid_fare_amount": F.col("fare_amount").isNotNull() & (F.col("fare_amount") < 2.5),
    
}

_CANONICAL: Tuple[Tuple[str, T.DataType], ...] = (
    ("vendor_id", T.StringType()),
    ("pickup_datetime", T.TimestampType()),
    ("dropoff_datetime", T.TimestampType()),
    ("passenger_count", T.LongType()),
    ("trip_distance", T.DoubleType()),
    ("rate_code", T.LongType()),
    ("store_and_fwd_flag", T.StringType()),
    ("pickup_longitude", T.DoubleType()),
    ("pickup_latitude", T.DoubleType()),
    ("dropoff_longitude", T.DoubleType()),
    ("dropoff_latitude", T.DoubleType()),
    ("PULocationID", T.LongType()),
    ("DOLocationID", T.LongType()),
    ("payment_type", T.StringType()),
    ("fare_amount", T.DoubleType()),
    ("surcharge", T.DoubleType()),
    ("mta_tax", T.DoubleType()),
    ("tip_amount", T.DoubleType()),
    ("tolls_amount", T.DoubleType()),
    ("improvement_surcharge", T.DoubleType()),
    ("congestion_surcharge", T.DoubleType()),
    ("airport_fee", T.DoubleType()),
    ("cbd_congestion_fee", T.DoubleType()),
    ("total_amount", T.DoubleType()),
)

_LINEAGE_COL = "_input_file"


@dataclass(frozen=True)
class DQResult:
    total_rows: int
    bad_rows: int
    bad_by_rule: Dict[str, int]


def _normalize_one(df: DataFrame, *, keep_cols: Tuple[str, ...] = (_LINEAGE_COL,)) -> DataFrame:
    if "__index_level_0__" in df.columns:
        df = df.drop("__index_level_0__")

    cols = set(df.columns)
    is_new = ("tpep_pickup_datetime" in cols) or ("tpep_dropoff_datetime" in cols) or ("PULocationID" in cols)

    if is_new:
        rename = {
            "VendorID": "vendor_id",
            "tpep_pickup_datetime": "pickup_datetime",
            "tpep_dropoff_datetime": "dropoff_datetime",
            "RatecodeID": "rate_code",
            "extra": "surcharge",
            "Airport_fee": "airport_fee",
        }
        for src, dst in rename.items():
            if src in df.columns and dst not in df.columns:
                df = df.withColumnRenamed(src, dst)

        if "vendor_id" in df.columns:
            df = df.withColumn("vendor_id", F.col("vendor_id").cast("string"))
        if "PULocationID" in df.columns:
            df = df.withColumn("PULocationID", F.col("PULocationID").cast("long"))
        if "DOLocationID" in df.columns:
            df = df.withColumn("DOLocationID", F.col("DOLocationID").cast("long"))
        if "rate_code" in df.columns:
            df = df.withColumn("rate_code", F.col("rate_code").cast("long"))
        if "payment_type" in df.columns:
            df = df.withColumn("payment_type", F.col("payment_type").cast("string"))
        if "airport_fee" in df.columns:
            df = df.withColumn("airport_fee", F.col("airport_fee").cast("double"))
        if "congestion_surcharge" in df.columns:
            df = df.withColumn("congestion_surcharge", F.col("congestion_surcharge").cast("double"))
    else:
        if "vendor_id" in df.columns:
            df = df.withColumn("vendor_id", F.col("vendor_id").cast("string"))
        if "rate_code" in df.columns:
            df = df.withColumn("rate_code", F.col("rate_code").cast("long"))
        if "payment_type" in df.columns:
            df = df.withColumn("payment_type", F.col("payment_type").cast("string"))
        if "pickup_datetime" in df.columns:
            df = df.withColumn("pickup_datetime", F.to_timestamp("pickup_datetime"))
        if "dropoff_datetime" in df.columns:
            df = df.withColumn("dropoff_datetime", F.to_timestamp("dropoff_datetime"))

    for name, dtype in _CANONICAL:
        if name not in df.columns:
            df = df.withColumn(name, F.lit(None).cast(dtype))
        else:
            df = df.withColumn(name, F.col(name).cast(dtype))

    select_cols = [c for c, _ in _CANONICAL] + [c for c in keep_cols if c in df.columns]
    return df.select(*select_cols)


def _add_yyyymm_from_path(df: DataFrame, path_col: str = _LINEAGE_COL) -> DataFrame:
    y = F.regexp_extract(F.col(path_col), r"yellow_tripdata_(\d{4})-(\d{2})\.parquet", 1)
    m = F.regexp_extract(F.col(path_col), r"yellow_tripdata_(\d{4})-(\d{2})\.parquet", 2)

    return (
        df.withColumn("YYYY", F.when(y != "", y.cast("int")))
          .withColumn("YYYYMM", F.when((y != "") & (m != ""), F.concat(y, m).cast("int")))
    )

"""
def _add_duration_min(df: DataFrame) -> DataFrame:
    return df.withColumn(
        "duration_min",
        (F.unix_timestamp("dropoff_datetime") - F.unix_timestamp("pickup_datetime")) / F.lit(60.0),
    )
"""

def _dq_check(df: DataFrame) -> Tuple[DataFrame, DQResult]:


    # make every rule strictly boolean (NULL -> False)
    rules_bool = {name: F.coalesce(cond.cast("boolean"), F.lit(False)) for name, cond in BASE_RULES.items()}
    dq_reason_raw = F.array(*[F.when(cond, F.lit(name)) for name, cond in rules_bool.items()])
    # remove nulls safely (NO array_remove with NULL)
    dq_reason = F.expr("filter(dq_reason_raw, x -> x is not null)")
    out = (
        df.withColumn("dq_reason_raw", dq_reason_raw)
          .withColumn("dq_reason", dq_reason)
          .withColumn("dq_is_bad", F.size(F.col("dq_reason")) > 0)
          .drop("dq_reason_raw")
    )
    agg_exprs = [F.count_if(cond).alias(name) for name, cond in rules_bool.items()]
    agg_exprs.append(F.count("*").alias("total_rows"))
    agg_exprs.append(F.count_if(F.col("dq_is_bad")).alias("bad_rows"))
    
    summary_row = out.select(*agg_exprs).collect()[0]
    
    bad_by_rule = {name: summary_row[name] for name in rules_bool.keys()}
    
    return out, DQResult(
        total_rows=summary_row["total_rows"],
        bad_rows=summary_row["bad_rows"],
        bad_by_rule=bad_by_rule,
    )


def _list_parquet_files(spark: SparkSession, folder: str, *, recursive: bool) -> List[str]:
    from pyspark.dbutils import DBUtils
    dbutils = DBUtils(spark)
    
    files = []
    # 定义一个内部递归函数
    def _scan(path: str):
        for file_info in dbutils.fs.ls(path):
            if file_info.isDir():
                if recursive:
                    _scan(file_info.path)
            elif file_info.path.endswith(".parquet"):
                files.append(file_info.path)
    
    _scan(folder)
    return sorted(files)


def _merge_dq(agg: DQResult, part: DQResult) -> DQResult:
    merged_rules: Dict[str, int] = dict(agg.bad_by_rule)
    for k, v in part.bad_by_rule.items():
        merged_rules[k] = merged_rules.get(k, 0) + v
    return DQResult(
        total_rows=agg.total_rows + part.total_rows,
        bad_rows=agg.bad_rows + part.bad_rows,
        bad_by_rule=merged_rules,
    )


def process_single_parquet_file(
    spark: SparkSession,
    parquet_path: str,
    *,
    fail_on_bad_filename: bool = False,
    keep_lineage_col: bool = False,
) -> Tuple[DataFrame, DQResult]:
    raw = spark.read.parquet(parquet_path).withColumn(_LINEAGE_COL, F.col("_metadata.file_path"))
    norm = _normalize_one(raw, keep_cols=(_LINEAGE_COL,))
    with_dates = _add_yyyymm_from_path(norm, path_col=_LINEAGE_COL)
    #with_dates = _add_duration_min(with_dates)

    if fail_on_bad_filename:
        bad_name_cnt = with_dates.filter(F.col("YYYY").isNull()).count()
        if bad_name_cnt > 0:
            raise ValueError(
                f"{bad_name_cnt} rows in {parquet_path!r} have paths not matching "
                f"'yellow_tripdata_YYYY-MM.parquet' (could not derive YYYY/YYYYMM)."
            )

    if not keep_lineage_col:
        with_dates = with_dates.drop(_LINEAGE_COL)

    dq_df, dq = _dq_check(with_dates)
    #with_audit = with_dates.withColumn("_load_timestamp", F.current_timestamp()) 
    with_audit = dq_df.withColumn("_load_timestamp", F.current_timestamp()) 

    return with_audit, dq


def merge_yellow_tripdata_parquet_folder(
    spark: SparkSession,
    folder: str,
    *,
    recursive: bool = False,
    fail_on_bad_filename: bool = False,
    run_dq: bool = True,
    keep_lineage_col: bool = False,
) -> Tuple[DataFrame, Optional[DQResult]]:
    """
    Bulk read (single scan). Still returns one DataFrame + rolled-up DQ.
    Prefer write_yellow_tripdata_to_delta for append-per-file loads.
    """
    reader = spark.read.option("recursiveFileLookup", "true" if recursive else "false")
    raw = reader.parquet(folder).withColumn(_LINEAGE_COL, F.col("_metadata.file_path"))

    norm = _normalize_one(raw, keep_cols=(_LINEAGE_COL,))
    with_dates = _add_yyyymm_from_path(norm, path_col=_LINEAGE_COL)
    #with_dates = _add_duration_min(with_dates)

    if fail_on_bad_filename:
        bad_name_cnt = with_dates.filter(F.col("YYYY").isNull()).count()
        if bad_name_cnt > 0:
            raise ValueError(
                f"{bad_name_cnt} rows have paths not matching "
                f"'yellow_tripdata_YYYY-MM.parquet' (could not derive YYYY/YYYYMM)."
            )

    if not keep_lineage_col:
        with_dates = with_dates.drop(_LINEAGE_COL)

    if run_dq:
        dq_df, dq = _dq_check(with_dates)
        return dq_df, dq

    return with_dates, None


def write_yellow_tripdata_to_delta(
    spark: SparkSession,
    folder: str,
    delta_target: str,
    *,
    recursive: bool = False,
    fail_on_bad_filename: bool = False,
    write_mode: str = "append",
    partition_by: Optional[Tuple[str, ...]] = ("YYYYMM",),
    merge_schema: bool = True,
    keep_lineage_col: bool = False,
) -> DQResult:
    """
    For each Parquet file under folder: normalize → YYYY/YYYYMM → duration → dq_is_bad/dq_reason
    → append rows to one Delta table (no single giant union in memory).

    write_mode: use \"append\" (default). If you pass \"overwrite\", only the first file write
    uses overwrite; remaining files use append so the table is not wiped each iteration.
    """
    files = _list_parquet_files(spark, folder, recursive=recursive)
    if not files:
        raise FileNotFoundError(f"No .parquet files found under: {folder!r}")

    agg = DQResult(total_rows=0, bad_rows=0, bad_by_rule={})
    expected_row_count = 0

    for i, path in enumerate(files):
        df, dq_part = process_single_parquet_file(
            spark,
            path,
            fail_on_bad_filename=fail_on_bad_filename,
            keep_lineage_col=keep_lineage_col,
        )
        agg = _merge_dq(agg, dq_part)
        expected_row_count += dq_part.total_rows

        print(f"File loaded: {path.split('/')[-1]} | Rows added: {dq_part.total_rows}")

        current_yyyymm = df.select("YYYYMM").first()[0]

        mode = write_mode if i == 0 else "append"
        
        if mode == "overwrite":
            writer = (
                df.write.format("delta")
                .mode(mode)
                .option("replaceWhere", f"YYYYMM = {current_yyyymm}")
                .option("mergeSchema", str(merge_schema).lower())
            )
        else: 
            writer = (
                df.write.format("delta")
                .mode(mode)
                .option("mergeSchema", str(merge_schema).lower())
            )

        if partition_by:
            writer = writer.partitionBy(*partition_by)

        writer.saveAsTable(delta_target)

    # 5. 整个循环结束后，进行对账检查 (Reconciliation)

    print("\n" + "="*40)
    print("DATA QUALITY RECONCILIATION")
    
    res_df = spark.sql(f"SELECT COUNT(*) AS total FROM {delta_target}")

    # 2. 从 DataFrame 中取出那个数字
    # .collect()[0][0] 表示取第一行、第一列的值
    actual_row_count = res_df.collect()[0][0]

    print(f"Total Expected Rows (Sum of Parquet): {expected_row_count}")
    print(f"Total Actual Rows (Delta Table):      {actual_row_count}")

    if expected_row_count == actual_row_count:
        print("DQ Check Passed: 数据加载完整，行数匹配！")
    else:
        diff = expected_row_count - actual_row_count
        print(f"DQ Check Failed: 发现数据差异！差异行数: {diff}")
        # 如果差异不可接受，可以在这里抛出异常中止程序
        # raise ValueError(f"Data loss detected! Diff: {diff}")

    return agg

In [0]:
%sql
SELECT 
    *,
    -- 标记具体的错误类型
    CASE 
        WHEN pickup_datetime IS NULL THEN 'missing_pickup'
        WHEN dropoff_datetime IS NULL THEN 'missing_dropoff'
        WHEN dropoff_datetime < pickup_datetime THEN 'dropoff_before_pickup'
        WHEN passenger_count < 0 THEN 'passenger_negative'
        WHEN trip_distance < 0.1 THEN 'trip_too_short'
        WHEN total_amount < 0 THEN 'total_amount_negative'
        WHEN YYYYMM < 190001 OR YYYYMM > 300012 THEN 'invalid_yyyymm'
        WHEN fare_amount < 2.5 THEN 'invalid_fare'
        ELSE 'Unknown'
    END as violation_type
FROM process_silver.delta_yellow_taxi
WHERE 
    pickup_datetime IS NULL 
    OR dropoff_datetime IS NULL
    OR dropoff_datetime < pickup_datetime
    OR passenger_count < 0
    OR trip_distance < 0.1
    OR total_amount < 0
    OR (YYYYMM IS NOT NULL AND (YYYYMM < 190001 OR YYYYMM > 300012))
    OR fare_amount < 2.5;

In [0]:
%sql
SELECT *,
    CASE WHEN pickup_datetime IS NULL THEN 'missing_pickup_datetime' END AS r1,
    CASE WHEN dropoff_datetime IS NULL THEN 'missing_dropoff_datetime' END AS r2,
    CASE WHEN pickup_datetime IS NOT NULL 
          AND dropoff_datetime IS NOT NULL 
          AND dropoff_datetime < pickup_datetime 
         THEN 'dropoff_before_pickup' END AS r3,
    CASE WHEN passenger_count < 0 THEN 'passenger_count_negative' END AS r4,
    CASE WHEN trip_distance < 0.1 THEN 'trip_distance_too_small' END AS r5,
    CASE WHEN total_amount < 0 THEN 'total_amount_negative' END AS r6,
    CASE WHEN YYYYMM IS NOT NULL AND (YYYYMM < 190001 OR YYYYMM > 300012)
         THEN 'invalid_YYYYMM' END AS r7,
    CASE WHEN fare_amount < 2.5 THEN 'invalid_fare_amount' END AS r8
FROM process_silver.delta_yellow_taxi;

In [0]:
%sql
SELECT *
FROM process_silver.delta_yellow_taxi 
WHERE 
    pickup_datetime IS NULL
    OR dropoff_datetime IS NULL
    OR (pickup_datetime IS NOT NULL AND dropoff_datetime IS NOT NULL AND dropoff_datetime < pickup_datetime)
    OR passenger_count < 0
    OR trip_distance < 0.1
    OR total_amount < 0
    OR (YYYYMM IS NOT NULL AND (YYYYMM < 190001 OR YYYYMM > 300012))
    OR fare_amount < 2.5;

In [0]:
%sql
DROP TABLE workspace.default.delta_yellow_taxi; 

In [0]:
display(spark.sql("SHOW TABLES"))

In [0]:


dq = write_yellow_tripdata_to_delta(
    spark,
    #folder=yellow_path,
    folder="/Volumes/nyc/default/yellowtaxi/yellow_tripdata_2010-01.parquet", 
    delta_target=delta_target,
    write_mode="overwrite",
    partition_by=(target_yellow_partition,),
)
print(dq)

In [0]:
%sql
select * from process_silver.delta_yellow_taxi where dq_reason is not null limit 19; 

In [0]:
spark.table("delta_yellow_taxi").printSchema()

In [0]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Optional, Tuple

import re
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T


# Canonical (merged) output columns (includes both GPS-era and ZoneID-era fields)
_CANONICAL: Tuple[Tuple[str, T.DataType], ...] = (
    ("vendor_id", T.StringType()),
    ("pickup_datetime", T.TimestampType()),
    ("dropoff_datetime", T.TimestampType()),
    ("passenger_count", T.LongType()),
    ("trip_distance", T.DoubleType()),
    ("rate_code", T.LongType()),
    ("store_and_fwd_flag", T.StringType()),
    ("pickup_longitude", T.DoubleType()),
    ("pickup_latitude", T.DoubleType()),
    ("dropoff_longitude", T.DoubleType()),
    ("dropoff_latitude", T.DoubleType()),
    ("PULocationID", T.LongType()),
    ("DOLocationID", T.LongType()),
    ("payment_type", T.StringType()),  # string so old string + new numeric codes can coexist
    ("fare_amount", T.DoubleType()),
    ("surcharge", T.DoubleType()),     # 2010/2013 "surcharge"; 2016/2026 "extra" mapped here
    ("mta_tax", T.DoubleType()),
    ("tip_amount", T.DoubleType()),
    ("tolls_amount", T.DoubleType()),
    ("improvement_surcharge", T.DoubleType()),
    ("congestion_surcharge", T.DoubleType()),
    ("airport_fee", T.DoubleType()),
    ("cbd_congestion_fee", T.DoubleType()),
    ("total_amount", T.DoubleType()),
)


@dataclass(frozen=True)
class DQResult:
    total_rows: int
    bad_rows: int
    bad_by_rule: Dict[str, int]


def _normalize_one(df: DataFrame) -> DataFrame:
    # drop pandas index column if present
    if "__index_level_0__" in df.columns:
        df = df.drop("__index_level_0__")

    cols = set(df.columns)
    is_new = ("tpep_pickup_datetime" in cols) or ("tpep_dropoff_datetime" in cols) or ("PULocationID" in cols)

    if is_new:
        rename = {
            "VendorID": "vendor_id",
            "tpep_pickup_datetime": "pickup_datetime",
            "tpep_dropoff_datetime": "dropoff_datetime",
            "RatecodeID": "rate_code",
            "extra": "surcharge",
            "Airport_fee": "airport_fee",  # 2026 casing
        }
        for src, dst in rename.items():
            if src in df.columns and dst not in df.columns:
                df = df.withColumnRenamed(src, dst)

        # align types (int vs long, etc.)
        if "vendor_id" in df.columns:
            df = df.withColumn("vendor_id", F.col("vendor_id").cast("string"))
        if "PULocationID" in df.columns:
            df = df.withColumn("PULocationID", F.col("PULocationID").cast("long"))
        if "DOLocationID" in df.columns:
            df = df.withColumn("DOLocationID", F.col("DOLocationID").cast("long"))
        if "rate_code" in df.columns:
            df = df.withColumn("rate_code", F.col("rate_code").cast("long"))
        if "payment_type" in df.columns:
            df = df.withColumn("payment_type", F.col("payment_type").cast("string"))
        if "airport_fee" in df.columns:
            df = df.withColumn("airport_fee", F.col("airport_fee").cast("double"))
        if "congestion_surcharge" in df.columns:
            df = df.withColumn("congestion_surcharge", F.col("congestion_surcharge").cast("double"))

    else:
        # old schema: vendor_id/pickup_datetime/dropoff_datetime are strings
        if "vendor_id" in df.columns:
            df = df.withColumn("vendor_id", F.col("vendor_id").cast("string"))
        if "rate_code" in df.columns:
            df = df.withColumn("rate_code", F.col("rate_code").cast("long"))
        if "payment_type" in df.columns:
            df = df.withColumn("payment_type", F.col("payment_type").cast("string"))

        # parse to timestamp (Spark can usually infer common timestamp strings)
        if "pickup_datetime" in df.columns:
            df = df.withColumn("pickup_datetime", F.to_timestamp("pickup_datetime"))
        if "dropoff_datetime" in df.columns:
            df = df.withColumn("dropoff_datetime", F.to_timestamp("dropoff_datetime"))

    # guarantee canonical cols exist with canonical types
    for name, dtype in _CANONICAL:
        if name not in df.columns:
            df = df.withColumn(name, F.lit(None).cast(dtype))
        else:
            df = df.withColumn(name, F.col(name).cast(dtype))

    return df.select([c for c, _ in _CANONICAL])


def _add_yyyymm_from_filename(df: DataFrame, input_path_col: str = "_input_file") -> DataFrame:
    # expects file names like yellow_tripdata_2010-01.parquet
    # extract yyyy and mm from full path
    y = F.regexp_extract(F.col(input_path_col), r"yellow_tripdata_(\d{4})-(\d{2})\.parquet", 1)
    m = F.regexp_extract(F.col(input_path_col), r"yellow_tripdata_(\d{4})-(\d{2})\.parquet", 2)

    yyyy = F.when(y != "", y.cast("int"))
    yyyymm = F.when((y != "") & (m != ""), F.concat(y, m).cast("int"))

    return (
        df.withColumn("YYYY", yyyy)
          .withColumn("YYYYMM", yyyymm)
    )


def _dq_check(df: DataFrame) -> Tuple[DataFrame, DQResult]:
    """
    Adds `dq_is_bad` + `dq_reason` (array of rule names). Returns (df_with_dq, summary).
    """
    rules = {
        "missing_pickup_datetime": F.col("pickup_datetime").isNull(),
        "missing_dropoff_datetime": F.col("dropoff_datetime").isNull(),
        "dropoff_before_pickup": (F.col("pickup_datetime").isNotNull())
                                & (F.col("dropoff_datetime").isNotNull())
                                & (F.col("dropoff_datetime") < F.col("pickup_datetime")),
        "passenger_count_negative": F.col("passenger_count") < 0,
        "trip_distance_negative": F.col("trip_distance") < 0.1,
        "total_amount_negative": F.col("total_amount") < 0,
        "invalid_YYYYMM": F.col("YYYYMM").isNotNull() & ((F.col("YYYYMM") < 190001) | (F.col("YYYYMM") > 300012)),
        "invalid_fare_amount": F.col("fare_amount") < 2.5,   # filter out the fare amount less than Starting price 
        "invalid_duration_min": (F.col("duration_min") < 2.0) | (F.col("duration_min") > 300)

    }

    dq_reason = F.array_remove(
        F.array(*[F.when(cond, F.lit(name)) for name, cond in rules.items()]),
        F.lit(None),
    )
    out = df.withColumn("dq_reason", dq_reason).withColumn("dq_is_bad", F.size("dq_reason") > 0)

    total = out.count()
    bad = out.filter(F.col("dq_is_bad")).count()

    bad_by_rule = {
        name: out.filter(cond).count()
        for name, cond in rules.items()
    }

    return out, DQResult(total_rows=total, bad_rows=bad, bad_by_rule=bad_by_rule)


def merge_yellow_tripdata_parquet_folder(
    spark: SparkSession,
    folder: str,
    *,
    recursive: bool = False,
    fail_on_bad_filename: bool = False,
    run_dq: bool = True,
) -> Tuple[DataFrame, Optional[DQResult]]:
    """
    Merge 2010/2013/2016/2026-style parquet files living in one folder, normalize schema,
    drop __index_level_0__ if present, and add YYYY and YYYYMM derived from the filename.

    Returns (merged_df, dq_result). If run_dq=False, dq_result is None.

    Notes:
    - Adds `_input_file` for lineage; drop it if you don't want it.
    - DQ adds `dq_is_bad` and `dq_reason` columns if run_dq=True.
    """
    reader = spark.read.option("recursiveFileLookup", "true" if recursive else "false")

    # read everything; include the file name for deriving YYYY/YYYMM
    # raw = reader.parquet(folder).withColumn("_input_file", F.input_file_name())
    raw = reader.parquet(folder).withColumn("_input_file", F.col("_metadata.file_path"))

    # normalize schema + add YYYY/YYYMM from filename
    norm = _normalize_one(raw)
    norm = norm.join(raw.select(F.monotonically_increasing_id().alias("__rid"), "_input_file"), how="inner") \
               if "_input_file" not in norm.columns else norm
    # safer: just re-add input file column from raw using same row order is not guaranteed,
    # so instead keep _input_file before selecting canonical:
    # (We already selected canonical in _normalize_one, so re-attach directly from raw by column if present.)
    # If Spark preserved _input_file through normalization, this is a no-op.

    # If _input_file got dropped by select, re-create it from input_file_name() again:
    if "_input_file" not in norm.columns:
        #norm = norm.withColumn("_input_file", F.input_file_name())
        norm = norm.withColumn("_input_file", F.col("_metadata.file_path"))

    with_dates = _add_yyyymm_from_filename(norm, input_path_col="_input_file")

    if fail_on_bad_filename:
        # rows where filename didn't match pattern -> YYYY is null
        bad_name_cnt = with_dates.filter(F.col("YYYY").isNull()).count()
        if bad_name_cnt > 0:
            raise ValueError(
                f"{bad_name_cnt} rows have filenames not matching pattern "
                f"'yellow_tripdata_YYYY-MM.parquet' (could not derive YYYY/YYYMM)."
            )

    if run_dq:
        dq_df, dq = _dq_check(with_dates)
        return dq_df, dq

    return with_dates, None

In [0]:
# 1) Basic usage (one folder of parquet files)
df, dq = merge_yellow_tripdata_parquet_folder(
    spark,
    yellow_path,
    recursive=False,     # set True if files are in subfolders
    run_dq=True,
)

# 2) Keep only “good” rows
good = df.filter(~F.col("dq_is_bad"))

# 3) Or keep everything but review DQ summary


# 4) Write merged output
# good.write.mode("overwrite").parquet("/path/to/output/merged_yellow_tripdata")

In [0]:
display(df.select("YYYY").distinct().limit(5))

In [0]:
from pyspark.sql import functions as F

def load_file_by_file(folder_path):
    # 1. 获取文件夹下所有的 parquet 文件路径
    files = dbutils.fs.ls(folder_path)
    file_paths = [f.path for f in files if f.path.endswith(".parquet")]
    
    df_list = []
    
    for p in file_paths:
        # 2. 逐个文件读取，此时不会触发 Schema 合并冲突
        temp_df = spark.read.format("parquet").load(p).select("*", F.col("_metadata.file_path").alias("raw_path"))
        
        # 3. 立即进行标准化转换
        # 统一列名（兼容 2010, 2013, 2026）和数据类型
        # 这里使用表达式来灵活处理不同年份的列名
        
        # 检查是否是 2013/2026 的列名风格
        is_new_style = "VendorID" in temp_df.columns
        
        if is_new_style:
            temp_df = temp_df.selectExpr(
                "cast(VendorID as string) as vendor_id",
                "cast(tpep_pickup_datetime as string) as pickup_datetime",
                "cast(passenger_count as long) as passenger_count",
                "cast(RatecodeID as string) as rate_code",
                "cast(payment_type as string) as payment_type",
                "fare_amount",
                "total_amount",
                "raw_path"
            )
        else:
            # 2010 年风格
            temp_df = temp_df.selectExpr(
                "cast(vendor_id as string) as vendor_id",
                "cast(pickup_datetime as string) as pickup_datetime",
                "cast(passenger_count as long) as passenger_count",
                "cast(rate_code as string) as rate_code",
                "cast(payment_type as string) as payment_type",
                "fare_amount",
                "total_amount",
                "raw_path"
            )
            
        # 4. 提取日期
        temp_df = (temp_df
            .withColumn("date_parts", F.regexp_extract(F.col("raw_path"), r"(\d{4})[-_](\d{2})", 0))
            .withColumn("YYYYMM", F.regexp_replace(F.col("date_parts"), r"[-_]", "").cast("integer"))
            .withColumn("YYYY", F.substring(F.col("date_parts"), 1, 4).cast("integer"))
            .drop("raw_path", "date_parts")
        )
        
        df_list.append(temp_df)
    
    # 5. 使用 unionByName 将所有单文件 DataFrame 合并
    # 因为我们已经在循环里把类型 cast 统一了，所以这里的 union 会非常丝滑
    from functools import reduce
    final_df = reduce(lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True), df_list)
    
    return final_df

# 调用
all_years_df = load_file_by_file(yellow_path)

##### Add Checkpoint

In [0]:
# Set path for checkpoint 

#spark.sparkContext.setCheckpointDir("/Volumes/nyc/default/checkpoint")

from pyspark.sql import functions as F 
from functools import reduce 

def load_and_merge_massive_files(file_paths): 
    # file_paths includes all parquet file in the path 

    total_files = len(file_paths)
    batch_size = 10 

    final_df = None 

    for i, path in enumerate(file_paths): 
        # load file to a temp_df 
        temp_df = spark.read.format("parquet").load(path).select("*", F.col("_metadata.file_path").alias("raw_path"))

        is_new_style = "VendorID" in temp_df.columns
        
        if is_new_style:
            temp_df = temp_df.selectExpr(
                "cast(VendorID as string) as vendor_id",
                "cast(tpep_pickup_datetime as string) as pickup_datetime",
                "cast(passenger_count as long) as passenger_count",
                "cast(RatecodeID as string) as rate_code",
                "cast(payment_type as string) as payment_type",
                "fare_amount",
                "total_amount",
                "raw_path"
            )
        else:
            # 2010 年风格
            temp_df = temp_df.selectExpr(
                "cast(vendor_id as string) as vendor_id",
                "cast(pickup_datetime as string) as pickup_datetime",
                "cast(passenger_count as long) as passenger_count",
                "cast(rate_code as string) as rate_code",
                "cast(payment_type as string) as payment_type",
                "fare_amount",
                "total_amount",
                "raw_path"
            )

        if final_df is None: 
            final_df = temp_df 
        else: 
            final_df = final_df.unionByName(temp_df, allowMissingColumns = True)

        if (i + 1) % batch_size == 0: 
            print(f"Process prograss: {i + 1} / {total_files} - loading checkpoint") 

            final_df = final_df.localCheckpoint(eager=True) 

    return final_df 

# get all paths 
all_files = [f.path for f in dbutils.fs.ls(yellow_path) if f.path.endswith(".parquet")] 

result_df = load_and_merge_massive_files(all_files) 

display(result_df.limit(5))



In [0]:
display(all_years_df.select("YYYYMM").distinct())

In [0]:
%sql
select * from parquet.`/Volumes/nyc/default/yellowtaxi/yellow_tripdata_2010-02.parquet` limit 10; 

In [0]:
df = (spark.read.format("parquet").load("/Volumes/nyc/default/yellowtaxi/yellow_tripdata_2013-01.parquet")) 

display(df.limit(5))

In [0]:
df = (spark.read.format("parquet").load("/Volumes/nyc/default/yellowtaxi/yellow_tripdata_2010-01.parquet")) 

display(df.limit(5))

In [0]:
df = (spark.read.format("parquet").load("/Volumes/nyc/default/yellowtaxi/yellow_tripdata_2010-02.parquet")) 

display(df.limit(5))

In [0]:
df = (spark.read.format("parquet").load("/Volumes/nyc/default/yellowtaxi/yellow_tripdata_2026-01.parquet")) 

df.printSchema()

In [0]:
df = (spark.read.format("parquet").load(yellow_path)) 

df.printSchema()

In [0]:
def load_with_filename_date(path): 
    from pyspark.sql import functions as F
    df = (
        spark.read
        .format("parquet")
        #.option("mergeSchema", "true")
        .load(path)
        .select("*", F.col("_metadata.file_path").alias("raw_path"))
    )

    df = (df
        .withColumn("vendor_id", F.col("VendorID").cast("string"))
        .withColumn("rate_code", F.col("rate_code").cast("string"))
        .withColumn("payment_type", F.col("payment_type").cast("string"))
        .withColumn("passenger_count", F.col("passenger_count").cast("long"))
    )
    
    # 3. 提取路径中的 YYYYMM（从文件路径元数据获取）
    df = (
        df.withColumn("date_parts", F.regexp_extract(F.col("raw_path"), r"(\d{4})[-_](\d{2})", 0)) 
           .withColumn("YYYYMM", F.regexp_replace(F.col("date_parts"), r"[-_]", "").cast("integer")) 
           .withColumn("YYYY", F.substring(F.col("date_parts"), 1, 4).cast("integer")) \
           .drop("file_path", "date_parts")
    )
    
    # 4. 丢弃像 __index_level_0__ 这样不需要的冗余列
    if "__index_level_0__" in df.columns:
        df = df.drop("__index_level_0__")

    return df

In [0]:
df_jan = load_with_filename_date("dbfs:/Volumes/nyc/default/yellowtaxi/yellow_tripdata_2010-01.parquet")
df_feb = load_with_filename_date("dbfs:/Volumes/nyc/default/yellowtaxi/yellow_tripdata_2010-02.parquet")

# 现在的 schema 是一致的了，可以合并
full_2010_df = df_jan.unionByName(df_feb, allowMissingColumns=True)

display(full_2010_df.limit(5))

###### 2013

In [0]:
from pyspark.sql import functions as F

def load_and_standardize_2013(path):
    # 1. 加载数据
    df = (
        spark.read.format("parquet").load(path)
        .select("*", F.col("_metadata.file_path").alias("raw_path"))
    )

    # 2. 映射列名并统一类型 (Mapping 2013 -> Standard)
    # 使用 selectExpr 可以非常方便地一边改名一边改类型
    standard_df = df.selectExpr(
        "cast(VendorID as string) as vendor_id",
        "cast(tpep_pickup_datetime as string) as pickup_datetime", # 统一转 string 或统一转 timestamp
        "cast(tpep_dropoff_datetime as string) as dropoff_datetime",
        "cast(passenger_count as long) as passenger_count",
        "trip_distance",
        "cast(RatecodeID as string) as rate_code",
        "store_and_fwd_flag",
        "PULocationID", # 2013年新增的地点 ID
        "DOLocationID",
        "cast(payment_type as string) as payment_type",
        "fare_amount",
        "mta_tax",
        "tip_amount",
        "tolls_amount",
        "total_amount",
        "raw_path" # 保留路径用于提取日期
    )

    # 3. 提取日期逻辑
    final_df = (
        standard_df
        .withColumn("date_parts", F.regexp_extract(F.col("raw_path"), r"(\d{4})[-_](\d{2})", 0))
        .withColumn("YYYYMM", F.regexp_replace(F.col("date_parts"), r"[-_]", "").cast("integer"))
        .withColumn("YYYY", F.substring(F.col("date_parts"), 1, 4).cast("integer"))
        .drop("raw_path", "date_parts")
    )

    return final_df

In [0]:
# 处理 2013 数据
full_2013_df = load_and_standardize_2013(yellow_2013_path)

display(full_2013_df.limit(5))

##### 2016

In [0]:
full_2016_df = load_and_standardize_2013(yellow_2016_path)

display(full_2016_df.limit(5))

##### 2026

In [0]:
from pyspark.sql import functions as F

def load_standardized_data(path, year_type="2026"):
    # 1. 基础读取
    df = spark.read.format("parquet").load(path).select("*", "_metadata.file_path")
    
    # 2. 定义映射关系 (目标列名: 源列名)
    # 以 2026 为例，我们把所有列映射回一套标准的“全量 Schema”
    if year_type == "2026":
        mapping = {
            "vendor_id": "VendorID",
            "pickup_datetime": "tpep_pickup_datetime",
            "dropoff_datetime": "tpep_dropoff_datetime",
            "passenger_count": "passenger_count",
            "trip_distance": "trip_distance",
            "rate_code": "RatecodeID",
            "store_and_fwd_flag": "store_and_fwd_flag",
            "payment_type": "payment_type",
            "fare_amount": "fare_amount",
            "extra": "extra",
            "mta_tax": "mta_tax",
            "tip_amount": "tip_amount",
            "tolls_amount": "tolls_amount",
            "total_amount": "total_amount",
            "improvement_surcharge": "improvement_surcharge",
            "congestion_surcharge": "congestion_surcharge",
            "airport_fee": "Airport_fee", # 修正大小写
            "cbd_fee": "cbd_congestion_fee" # 简写新列名
        }
    
    # 3. 动态构建 Select 语句
    selected_cols = [F.col(src).alias(tgt) for tgt, src in mapping.items() if src in df.columns]
    
    # 4. 执行转换与日期提取
    return (
        df.select(*selected_cols, F.col("file_path"))
        .withColumn("vendor_id", F.col("vendor_id").cast("string"))
        .withColumn("rate_code", F.col("rate_code").cast("string"))
        # 日期提取
        .withColumn("date_parts", F.regexp_extract(F.col("file_path"), r"(\d{4})[-_](\d{2})", 0))
        .withColumn("YYYYMM", F.regexp_replace(F.col("date_parts"), r"[-_]", "").cast("integer"))
        .withColumn("YYYY", F.substring(F.col("date_parts"), 1, 4).cast("integer"))
        .drop("file_path", "date_parts")
    )

In [0]:
from pyspark.sql import functions as F

def load_file_by_file(folder_path):
    # 1. 获取文件夹下所有的 parquet 文件路径
    files = dbutils.fs.ls(folder_path)
    file_paths = [f.path for f in files if f.path.endswith(".parquet")]
    
    df_list = []
    
    for p in file_paths:
        # 2. 逐个文件读取，此时不会触发 Schema 合并冲突
        temp_df = spark.read.format("parquet").load(p).select("*", F.col("_metadata.file_path").alias("raw_path"))
        
        # 3. 立即进行标准化转换
        # 统一列名（兼容 2010, 2013, 2026）和数据类型
        # 这里使用表达式来灵活处理不同年份的列名
        
        # 检查是否是 2013/2026 的列名风格
        is_new_style = "VendorID" in temp_df.columns
        
        if is_new_style:
            temp_df = temp_df.selectExpr(
                "cast(VendorID as string) as vendor_id",
                "cast(tpep_pickup_datetime as string) as pickup_datetime",
                "cast(passenger_count as long) as passenger_count",
                "cast(RatecodeID as string) as rate_code",
                "cast(payment_type as string) as payment_type",
                "fare_amount",
                "total_amount",
                "raw_path"
            )
        else:
            # 2010 年风格
            temp_df = temp_df.selectExpr(
                "cast(vendor_id as string) as vendor_id",
                "cast(pickup_datetime as string) as pickup_datetime",
                "cast(passenger_count as long) as passenger_count",
                "cast(rate_code as string) as rate_code",
                "cast(payment_type as string) as payment_type",
                "fare_amount",
                "total_amount",
                "raw_path"
            )
            
        # 4. 提取日期
        temp_df = (temp_df
            .withColumn("date_parts", F.regexp_extract(F.col("raw_path"), r"(\d{4})[-_](\d{2})", 0))
            .withColumn("YYYYMM", F.regexp_replace(F.col("date_parts"), r"[-_]", "").cast("integer"))
            .withColumn("YYYY", F.substring(F.col("date_parts"), 1, 4).cast("integer"))
            .drop("raw_path", "date_parts")
        )
        
        df_list.append(temp_df)
    
    # 5. 使用 unionByName 将所有单文件 DataFrame 合并
    # 因为我们已经在循环里把类型 cast 统一了，所以这里的 union 会非常丝滑
    from functools import reduce
    final_df = reduce(lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True), df_list)
    
    return final_df

# 调用
all_years_df = load_file_by_file("dbfs:/Volumes/nyc/default/yellowtaxi/")

In [0]:
full_2013_df = load_with_filename_date("dbfs:/Volumes/nyc/default/yellowtaxi/*2013*.parquet")

display(full_2013_df.limit(5))

In [0]:
from pyspark.sql.types import StringType, LongType, IntegerType, DateType, StructType, StructField, DoubleType, TimestampType

from pyspark.sql import functions as F 

schema_2010 = StructType([
    StructField("vendor_id", LongType(), True),
    StructField("pickup_datetime", StringType(), True), # 原始通常是String
    StructField("dropoff_datetime", StringType(), True),
    StructField("passenger_count", LongType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("pickup_longitude", DoubleType(), True),
    StructField("pickup_latitude", DoubleType(), True),
    StructField("rate_code", LongType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("dropoff_longitude", DoubleType(), True),
    StructField("dropoff_latitude", DoubleType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("surcharge", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("total_amount", DoubleType(), True)
])

#yellow_taxi_2010_raw_df = load_with_filename_date(yellow_2010_path, schema_2010)
yellow_taxi_2010_raw_df = load_with_filename_date(yellow_2010_path)

In [0]:
display(yellow_taxi_2010_raw_df.limit(5))

In [0]:
from pyspark.sql.types import StringType, LongType, IntegerType, DateType, StructType, StructField, DoubleType, TimestampType


schema_2013 = StructType([
    StructField("VendorID", LongType(), True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("passenger_count", LongType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", LongType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", IntegerType(), True),
    StructField("airport_fee", IntegerType(), True)
])


yellow_taxi_raw_df = (
    spark.read
    .format("parquet")
    .schema(schema_2013)
    .load(yellow_2013_path)
)

In [0]:
%pip install h3 

In [0]:
%pip install geopandas

In [0]:
%pip install folium

In [0]:
import h3
import geopandas as gpd
import folium
from shapely.geometry import mapping

# 1. 加载并转换坐标系 (确保列名匹配你的 Index: zone, LocationID, borough)
gdf = gpd.read_file("/Volumes/nyc/default/nyczone/taxi_zones/taxi_zones.shp").to_crs("EPSG:4326")

# 2. 定义 H3 转换函数 (使用你之前调通的质心方案，最稳健)
def get_h3_cell(geom, res=8):
    centroid = geom.centroid
    try:
        # 兼容 H3 4.x 和 3.x
        return h3.latlng_to_cell(centroid.y, centroid.x, res) if hasattr(h3, 'latlng_to_cell') else h3.geo_to_h3(centroid.y, centroid.x, res)
    except:
        return None

# 3. 生成数据
gdf['centroid_h3_cells'] = gdf['geometry'].apply(get_h3_cell)
# 这一步定义变量 h3_df_exploded
h3_df_exploded = gdf[['LocationID', 'borough', 'zone', 'centroid_h3_cells']].copy()


In [0]:
# 1. 将 Pandas DataFrame 转换为 Spark DataFrame
# 注意：转换时 Spark 会根据 Pandas 的数据自动推断 Schema
from pyspark.sql.window import Window 
from pyspark.sql.functions import row_number, col 

windowSpec = Window.partitionBy("VendorID","tpep_pickup_datetime", "tpep_dropoff_datetime").orderBy(col("tolls_amount").desc())

# 2. 增加行号并过滤
yellow_taxi_df = (
    yellow_taxi_raw_df
    .withColumn("row_num", row_number().over(windowSpec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

spark_h3_df = spark.createDataFrame(h3_df_exploded)


# 2. 执行 Join (现在两边都是 Spark DataFrame 了)
yellow_taxi_enriched_trips = yellow_taxi_df.join(
    spark_h3_df.select("LocationID", "centroid_h3_cells"), 
    on=yellow_taxi_df.PULocationID == spark_h3_df.LocationID, 
    how="inner"
)

# 3. 检查结果
# 注意：请确认字段名是 lpep_... 还是 tpep_...（根据你之前定义的 Schema 应该是 lpep）
yellow_taxi_enriched_trips.select("tpep_pickup_datetime", "PULocationID", "centroid_h3_cells").show(10)

#### Spatio-Temporal Hotspots 时间分桶

In [0]:
from pyspark.sql import functions as F

# 1. 提取时间特征并关联 H3 (假设你已经完成了 join 映射表的操作)
hotspot_base_df = yellow_taxi_enriched_trips.select(
    "centroid_h3_cells", 
    "tpep_pickup_datetime", 
    "VendorID"
).withColumn("pickup_hour", F.hour("tpep_pickup_datetime")) \
 .withColumn("day_of_week", F.dayofweek("tpep_pickup_datetime")) \
 .withColumn("day_name", F.date_format("tpep_pickup_datetime", "EEEE"))

# 2. 简单的质量检查
hotspot_base_df.select("tpep_pickup_datetime", "pickup_hour", "day_name").show(5)

#### Spatial-Temporal Aggregation

In [0]:

# 按 H3 格子、星期、小时进行聚合
hotspot_stats = hotspot_base_df.groupBy("centroid_h3_cells", "day_name", "pickup_hour") \
    .agg(F.count("*").alias("trip_count")) \
    .orderBy(F.desc("trip_count"))

# 显示最繁忙的 Top 10 “时空桶”
hotspot_stats.show(10)


#### Peak Detection

In [0]:
from pyspark.sql.window import Window

# 定义窗口：按地理格子分组，按订单量倒序排列
window_spec = Window.partitionBy("centroid_h3_cells").orderBy(F.desc("trip_count"))

# 找出每个格子单量最高的那一个小时
peak_analysis = hotspot_stats \
    .withColumn("rank", F.rank().over(window_spec)) \
    .filter(F.col("rank") == 1) \
    .select("centroid_h3_cells", "day_name", "pickup_hour", "trip_count")

# 看看纽约最繁忙的那些格子，它们的高峰通常发生在几点？
peak_analysis.orderBy(F.desc("trip_count")).show(10)


#### Seaborn Heatmap

In [0]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pyspark.sql import functions as F

# 1. 准备数据：为了画图清晰，我们先选出 Top 50 的 H3 热点格子
top_h3_cells = hotspot_stats.groupBy("centroid_h3_cells") \
    .agg(F.sum("trip_count").alias("total_trips")) \
    .orderBy(F.desc("total_trips")) \
    .limit(50) \
    .select("centroid_h3_cells")

# 2. 过滤并转化为 Pandas DataFrame
plot_data_spark = hotspot_stats.join(top_h3_cells, on="centroid_h3_cells", how="inner") \
    .select("centroid_h3_cells", "day_name", "pickup_hour", "trip_count")

plot_data_pd = plot_data_spark.toPandas()

# 3. 创建透视表 (Pivoting)：将数据整理成 Heatmap 格式
# 行是 H3 格子，列是小时
heatmap_data = plot_data_pd.pivot_table(
    index='centroid_h3_cells', 
    columns='pickup_hour', 
    values='trip_count', 
    aggfunc='sum'
).fillna(0) # 填充空值为 0

# 4. 绘图
plt.figure(figsize=(16, 8))
sns.heatmap(heatmap_data, cmap='YlOrRd', robust=True, xticklabels=2) # xticklabels=2 每两小时显示一个刻度
plt.title('2016 NYC Taxi Spatio-Temporal Hotspots (Top 50 H3 Cells)')
plt.xlabel('Pickup Hour (0-23)')
plt.ylabel('H3 Cell ID')
plt.xticks(rotation=0)
plt.show()

#### Kepler.gl / Folium

In [0]:
%pip install h3

In [0]:
import folium
import json
import h3
from pyspark.sql import functions as F

# 1. 准备数据：选择一个特定的时间段（例如：周五晚上 18:00 - 晚高峰）
friday_night_hotspots = hotspot_stats \
    .filter((F.col("day_name") == "Friday") & (F.col("pickup_hour") == 18)) \
    .orderBy(F.desc("trip_count")) \
    .limit(100) # 取 Top 100

plot_geo_pd = friday_night_hotspots.toPandas()

# 2. 初始化地图：以曼哈顿为中心
m = folium.Map(location=[40.7588, -73.9851], zoom_start=12, tiles='CartoDB positron')

# 3. 准备 GeoJSON 格式的六边形几何体
polygons = []
for index, row in plot_geo_pd.iterrows():
    h3_cell = row['centroid_h3_cells']
    trip_count = row['trip_count']
    
    # --- 关键修改处 ---
    # h3 v4 API 使用 cell_to_boundary
    coords = h3.cell_to_boundary(h3_cell) 
    # ------------------
    
    # Folium 需要的是 [[lng, lat], [lng, lat], ...] 的格式
    geojson_coords = [[lng, lat] for lat, lng in coords]
    geojson_coords.append(geojson_coords[0]) # 闭合多边形
    
    # 构造 GeoJSON 特征
    feature = {
        "type": "Feature",
        "geometry": {
            "type": "Polygon",
            "coordinates": [geojson_coords]
        },
        "properties": {
            "h3_cell": h3_cell,
            "trip_count": trip_count
        }
    }
    polygons.append(feature)

# 创建 GeoJSON 图层
geo_json_data = {"type": "FeatureCollection", "features": polygons}

# 4. 添加 Choropleth 层 (分级统计图)
folium.Choropleth(
    geo_data=geo_json_data,
    data=plot_geo_pd,
    columns=['centroid_h3_cells', 'trip_count'],
    key_on='feature.properties.h3_cell',
    fill_color='YlOrRd', # 黄-橙-红渐变
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Trip Count (Friday 18:00)'
).add_to(m)

# 显示地图
m

### 定义核心指标：每小时净收入 (Net Hourly Earnings)
在 2016 年，我们要衡量一个 H3 格子的“含金量”，不能只看总收入，要看时间产出比。

我们将计算：

Average Fare per Trip: 每一单平均能收多少钱。

Efficiency Index: 每分钟行程能赚多少钱（Fare / Trip Duration）。

In [0]:
from pyspark.sql import functions as F

# 1. 预处理：更加严谨的过滤逻辑
efficiency_base = yellow_taxi_enriched_trips \
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime")) \
    .withColumn("day_name", F.date_format("tpep_pickup_datetime", "EEEE")) \
    .withColumn("duration_min", 
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60
    ) \
    .withColumn("temp_eff", F.col("fare_amount") / F.col("duration_min")) \
    .filter(
        (F.col("duration_min") >= 2.0) & (F.col("duration_min") <= 180) & # 2分钟到3小时
        (F.col("trip_distance") > 0.1) &                                # 必须有行驶距离
        (F.col("temp_eff") < 15.0) &                                    # 过滤掉单分钟收益超过15刀的离群值
        (F.col("fare_amount") > 2.5)                                    # 必须超过纽约起步价
    ) \
    .drop("temp_eff") # 用完临时效率字段后删掉

# 2. 按 H3 和 小时 聚合

# revenue_stats = efficiency_base.groupBy("centroid_h3_cells") \
revenue_stats = efficiency_base.groupBy("centroid_h3_cells", "pickup_hour", "day_name") \
    .agg(
        F.count("*").alias("trip_count"),
        F.avg("fare_amount").alias("avg_fare"),
        # 使用 expr 或 F.col 确保计算逻辑清晰
        F.avg(F.col("fare_amount") / F.col("duration_min")).alias("earnings_per_min")
    )

# 3. 查看结果
revenue_stats.orderBy(F.desc("earnings_per_min")).show(10)

### 寻找“黄金格子” (The Golden Hexagons)
我们要找出那些订单量不低，且每分钟收益最高的区域。

In [0]:
# 过滤掉订单量太少的格子（比如一小时少于 10 单的），避免统计偏差
golden_grids = revenue_stats.filter("trip_count > 10") \
    .orderBy(F.desc("earnings_per_min"))

print("2016年 赚钱效率最高的 Top 10 H3 格子：")
golden_grids.show(10)

In [0]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. 将 Spark DataFrame 聚合到小时维度（平均所有格子）并转为 Pandas
hourly_trend_pd = revenue_stats.groupBy("pickup_hour") \
    .agg(F.avg("earnings_per_min").alias("avg_efficiency")) \
    .orderBy("pickup_hour") \
    .toPandas()

# 2. 绘图
plt.figure(figsize=(10, 5))
plt.plot(hourly_trend_pd['pickup_hour'], hourly_trend_pd['avg_efficiency'], marker='o', linestyle='-', color='green')
plt.title('2016 Average Earnings per Minute by Hour')
plt.xlabel('Hour of Day')
plt.ylabel('USD per Minute')
plt.grid(True, alpha=0.3)
plt.xticks(range(0, 24))
plt.show()

In [0]:
display(golden_grids.limit(15))

In [0]:
# 1. 重新聚合：按地理位置计算全周平均效率，确保每个 H3 只出现一次
geo_golden_grids = revenue_stats.groupBy("centroid_h3_cells") \
    .agg(
        F.sum("trip_count").alias("total_trips"),
        F.avg("earnings_per_min").alias("avg_earnings_per_min")
    ) \
    .filter("total_trips > 100") \
    .orderBy(F.desc("avg_earnings_per_min")) \
    .limit(15)

# 2. 转为 Pandas
top_grids_pd = geo_golden_grids.toPandas()

# 3. 绘图
plt.figure(figsize=(12, 6))

# 使用更直观的颜色（金色符合 Golden Grids 主题）
plt.bar(top_grids_pd['centroid_h3_cells'].astype(str), 
        top_grids_pd['avg_earnings_per_min'], 
        color='#FFD700', 
        edgecolor='#DAA520')

plt.title('2016 NYC Top 15 "Golden Grids" (Geographic Efficiency)', fontsize=14)
plt.ylabel('Avg Earnings per Minute ($/min)', fontsize=12)
plt.xlabel('H3 Cell ID', fontsize=12)

# 在柱子上标注具体数值，方便汇报
for i, val in enumerate(top_grids_pd['avg_earnings_per_min']):
    plt.text(i, val + 0.05, f'${val:.2f}', ha='center', va='bottom', fontsize=10)

plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [0]:
# 1. 将 Top 15 格子 ID 转为列表
top_15_ids = top_grids_pd['centroid_h3_cells'].tolist()

# 2. 创建一个标记地图
m_golden = folium.Map(location=[40.7588, -73.9851], zoom_start=11, tiles='CartoDB positron')

for index, row in top_grids_pd.iterrows():
    h3_id = row['centroid_h3_cells']
    eff = row['avg_earnings_per_min']
    
    # 获取中心点坐标
    lat, lng = h3.cell_to_latlng(h3_id)
    
    # 添加一个圆形标记，颜色越绿代表越赚钱
    folium.CircleMarker(
        location=[lat, lng],
        radius=10,
        popup=f"H3: {h3_id}\nEff: ${eff:.2f}/min",
        color='gold',
        fill=True,
        fill_color='gold',
        fill_opacity=0.7
    ).add_to(m_golden)

m_golden

In [0]:
# 1. 准备数据：确保包含 earnings_per_min 字段
# 建议直接从 revenue_stats（或你清洗后的 geo_efficiency_stats）中提取
plot_data_spark = (
    revenue_stats
        .filter("day_name = 'Friday' and pickup_hour = 18 and trip_count >100") 
        .orderBy(F.desc("earnings_per_min")) 
        .limit(100)       
) 

plot_geo_pd = plot_data_spark.toPandas()

# 验证一下字段是否存在
print(plot_geo_pd.columns)

In [0]:
import folium
import json
import h3
from pyspark.sql import functions as F

# 1. 准备数据：确保包含 earnings_per_min 字段
# 建议直接从 revenue_stats（或你清洗后的 geo_efficiency_stats）中提取
friday_night_hotspots =  (
    revenue_stats
        .filter("day_name = 'Friday' and pickup_hour = 18 and trip_count >100") 
        .orderBy(F.desc("earnings_per_min")) 
        .limit(100)       
) 

plot_geo_pd = friday_night_hotspots.toPandas()

# 2. 初始化地图：以曼哈顿为中心
m = folium.Map(location=[40.7588, -73.9851], zoom_start=12, tiles='CartoDB positron')

# 3. 准备 GeoJSON 格式的六边形几何体
polygons = []
for index, row in plot_geo_pd.iterrows():
    h3_cell = row['centroid_h3_cells']
    earnings_per_min = row['earnings_per_min']
    
    # --- 关键修改处 ---
    # h3 v4 API 使用 cell_to_boundary
    coords = h3.cell_to_boundary(h3_cell) 
    # ------------------
    
    # Folium 需要的是 [[lng, lat], [lng, lat], ...] 的格式
    geojson_coords = [[lng, lat] for lat, lng in coords]
    geojson_coords.append(geojson_coords[0]) # 闭合多边形
    
    # 构造 GeoJSON 特征
    feature = {
        "type": "Feature",
        "geometry": {
            "type": "Polygon",
            "coordinates": [geojson_coords]
        },
        "properties": {
            "h3_cell": h3_cell,
            "earnings_per_min": earnings_per_min
        }
    }
    polygons.append(feature)

# 创建 GeoJSON 图层
geo_json_data = {"type": "FeatureCollection", "features": polygons}

# 4. 添加 Choropleth 层 (分级统计图)
folium.Choropleth(
    geo_data=geo_json_data,
    data=plot_geo_pd,
    columns=['centroid_h3_cells', 'earnings_per_min'],
    key_on='feature.properties.h3_cell',
    fill_color='RdYlGn', # 黄-橙-红渐变
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Trip Count (Friday 18:00)'
).add_to(m)

# 显示地图
m

1. The Core Paradox: Volume vs. EfficiencyThe two maps highlight a fundamental mismatch in urban logistics: The busiest areas are often the least profitable for drivers.Map A: The "Busy" Hotspots (Trip Count)Observation: The heat is concentrated in Midtown and Lower Manhattan.Reality: This represents Demand. Friday at 18:00 (6 PM) is the peak of the "Rush Hour." Everyone is leaving offices or heading to dinner.The Trap: While there is a taxi on every corner and a passenger every second, the Congestion is at its maximum. Cars are moving at a walking pace (approx. 5–8 km/h).Map B: The "Golden" Hotspots (Earnings per Minute)Observation: The high-efficiency cells (purple/green) shift toward the edges of Manhattan, Brooklyn, and Airport corridors (JFK/LGA).Reality: This represents Profitability. These areas allow for "Flow."The Win: Drivers in these zones are likely picking up passengers heading for longer trips via highways (like the FDR Drive or the Midtown Tunnel). Because they can maintain a higher speed, the meter calculates fare based on distance rather than wait-time, which is significantly more lucrative.2. Why the "Center" Fails (The Efficiency Trap)In Data Engineering terms, your $earnings\_per\_min$ calculation exposed the Congestion Cost:Wait-Time Pricing: Taxis in 2016 charged a lower rate when moving slowly or stopped.Idle Waste: A driver in Midtown might spend 20 minutes to go 1 mile. Even with the base fare, the "return on time" is dismal.Fuel/Opportunity Cost: Frequent stopping and starting increases vehicle wear and prevents the driver from catching the next "big" fare.

Our 2016 Baseline analysis demonstrates that High Demand (Midtown Manhattan) does not correlate with High Driver Efficiency. Due to extreme congestion during Friday evening peak hours, the most profitable H3 cells are located along transit corridors and near airports, where higher vehicle velocity allows for better fare-to-time ratios. This creates a spatial 'Efficiency Map' that differs drastically from a 'Volume Map'.